# Perform segmentation and feature extraction using CellProfiler

NOTE: Plates with IDs that have spaces will still have spaces in the paths but the respective LoadData CSV with illum function paths will not have spaces due to `pe2loaddata` bug. This means that the LoadData from the previous IC module will be different (includes space).

## Import libraries

In [1]:
import argparse
import pathlib
import pprint

import sys

sys.path.append("../../utils")
import cp_parallel

# check if in a jupyter notebook
try:
    cfg = get_ipython().config
    in_notebook = True
except NameError:
    in_notebook = False

## Set paths and variables

In [2]:
#  directory where loaddata CSVs are located within the folder
loaddata_dir = pathlib.Path("./loaddata_csvs/").resolve(strict=True)

if not in_notebook:
    print("Running as script")

    parser = argparse.ArgumentParser(
        description="CellProfiler segmentation and feature extraction"
    )

    parser.add_argument(
        "--input_csv",
        type=str,
        required=True,
        help="Path to the LoadData CSV file to process images",
    )

    args = parser.parse_args()

    loaddata_csv = pathlib.Path(args.input_csv).resolve(strict=True)

else:
    print("Running in a notebook")

    loaddata_csv = pathlib.Path(
        f"{loaddata_dir}/Assay_Plate_1_3_loaddata_with_illum.csv"
    ).resolve(strict=True)

# set the run type for the parallelization
run_name = "analysis"

# set path for CellProfiler pipeline
path_to_pipeline = pathlib.Path("./pipeline/analysis_CHP134.cppipe").resolve(
    strict=True
)

# set main output dir for all plates if it doesn't exist
output_dir = pathlib.Path("./sqlite_outputs")
output_dir.mkdir(exist_ok=True)

Running in a notebook


## Create dictionary to process data

In [3]:
# Extract name from LoadData CSV path (drop loaddata suffix to avoid issues getting plate name)
name = loaddata_csv.stem
if name.endswith("_loaddata_with_illum"):
    name = name[: -len("_loaddata_with_illum")]

# create plate info dictionary with all parts of the CellProfiler CLI command to run in parallel
plate_info_dictionary = {
    name: {
        "path_to_loaddata": loaddata_csv,
        "path_to_output": output_dir / name,
        "path_to_pipeline": path_to_pipeline,
    }
}

# view the dictionary to assess that all info is added correctly
pprint.pprint(plate_info_dictionary, indent=4)

{   'Assay_Plate_1_3': {   'path_to_loaddata': PosixPath('/media/18tbdrive/1.Github_Repositories/pccma_repo1_screen_image_profiling/CHP-134_repo1_screen/2.feature_extraction/loaddata_csvs/Assay_Plate_1_3_loaddata_with_illum.csv'),
                           'path_to_output': PosixPath('sqlite_outputs/Assay_Plate_1_3'),
                           'path_to_pipeline': PosixPath('/media/18tbdrive/1.Github_Repositories/pccma_repo1_screen_image_profiling/CHP-134_repo1_screen/2.feature_extraction/pipeline/analysis_CHP134.cppipe')}}


## Perform segmentation and feature extraction (analysis)

Note: This code cell was not ran as we prefer to perform CellProfiler processing tasks via `sh` file (bash script) which is more stable.

In [4]:
cp_parallel.run_cellprofiler_parallel(
    plate_info_dictionary=plate_info_dictionary, run_name=run_name
)

All processes have been completed!
All results have been converted to log files!
